In [3]:
from __future__ import annotations
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Callable, Iterable, List, Optional, Tuple
from pyspark.sql import SparkSession, DataFrame, Row
from pyspark.sql import functions as F
from pyspark.sql import types as T


@dataclass(frozen=True)
class FieldSpec:
    name: str
    dtype: T.DataType
    required: bool = True
    coercion: Optional[Callable[[Any], Any]] = None
    validator: Optional[Callable[[Any], Optional[str]]] = None


def string(name: str, required: bool = True, *, coercion=None, validator=None) -> FieldSpec:
    return FieldSpec(name=name, dtype=T.StringType(), required=required, coercion=coercion, validator=validator)


def integer(name: str, required: bool = True, *, coercion=None, validator=None) -> FieldSpec:
    return FieldSpec(name=name, dtype=T.IntegerType(), required=required, coercion=coercion, validator=validator)


def build_struct_schema(fields: Iterable[FieldSpec]) -> T.StructType:
    return T.StructType([T.StructField(f.name, f.dtype, nullable=not f.required) for f in fields])


def create_spark_processor(name: str, fields: List[FieldSpec], *, input_col: str = 'raw_json', output_col: str = 'data'):
    struct_schema = build_struct_schema(fields)
    required_fields = [f.name for f in fields if f.required]

    def enforce_schema(self, df: DataFrame) -> DataFrame:
        parsed = df.withColumn(output_col, F.from_json(F.col(input_col), self.schema))
        missing_exprs = [F.col(f'{output_col}.{c}').isNull() for c in required_fields]
        is_missing = F.lit(False)
        for expr in missing_exprs:
            is_missing = is_missing | expr
        return (
            parsed
            .withColumn('is_schema_ok', (~is_missing) & F.col(output_col).isNotNull())
            .withColumn(
                'schema_error',
                F.when(F.col(output_col).isNull(), F.lit('JSON_PARSE_FAILED'))
                 .when(is_missing, F.lit('MISSING_REQUIRED_FIELD_OR_TYPE_CAST'))
                 .otherwise(F.lit(None))
            )
        )

    def _validate_and_normalize_py(record: Any) -> Tuple[Optional[dict], bool, Optional[str]]:
        if record is None:
            return None, False, 'NO_DATA'
        if isinstance(record, Row):
            rec = record.asDict(recursive=True)
        elif isinstance(record, dict):
            rec = record
        else:
            try:
                rec = dict(record)
            except Exception:
                return None, False, 'UNSUPPORTED_RECORD_TYPE'
        for f in fields:
            if f.required and (f.name not in rec or rec[f.name] is None):
                return None, False, f'MISSING:{f.name}'
        out = {}
        for f in fields:
            v = rec.get(f.name, None)
            if f.coercion is not None:
                try:
                    v = f.coercion(v)
                except Exception:
                    return None, False, f'COERCION_FAILED:{f.name}'
            if v is not None:
                try:
                    if isinstance(f.dtype, T.IntegerType):
                        v = int(v)
                    elif isinstance(f.dtype, T.StringType):
                        v = str(v)
                except Exception:
                    return None, False, f'TYPE_CAST_FAILED:{f.name}'
            if f.validator is not None:
                err = f.validator(v)
                if err:
                    return None, False, f'VALIDATION_FAILED:{f.name}:{err}'
            out[f.name] = v
        normalized = {k: (None if v is None else str(v)) for k, v in out.items()}
        return normalized, True, None

    validate_schema = T.StructType([
        T.StructField('normalized', T.MapType(T.StringType(), T.StringType(), valueContainsNull=True), True),
        T.StructField('is_valid', T.BooleanType(), False),
        T.StructField('error', T.StringType(), True),
    ])
    validate_udf = F.udf(_validate_and_normalize_py, validate_schema)

    def with_validation_udf(self, df: DataFrame) -> DataFrame:
        return (
            df.withColumn('validation', validate_udf(F.col(output_col)))
              .withColumn('is_valid', F.col('validation.is_valid'))
              .withColumn('validation_error', F.col('validation.error'))
        )

    return type(name, (object,), {
        'schema': struct_schema,
        'fields': fields,
        'enforce_schema': enforce_schema,
        'with_validation_udf': with_validation_udf,
    })


def normalize_email(v: Any) -> str:
    if v is None:
        raise ValueError('email is null')
    return str(v).strip().lower()


def validate_age(v: int | None) -> str | None:
    if v is None:
        return 'age is null'
    if v < 0 or v > 120:
        return 'age out of range [0, 120]'
    return None


def validate_country(v: str | None) -> str | None:
    allowed = {'PL', 'DE', 'FR', 'ES', 'IT', 'CZE'}
    if v is None:
        return 'country is null'
    if v not in allowed:
        return f'country not in {sorted(allowed)}'
    return None


if __name__ == '__main__':
    # Use Path('/content') for root_dir in Colab notebooks
    root_dir = Path('/content')
    input_path = root_dir / 'data' / 'dirty_users.jsonl'

    spark = (
        SparkSession.builder
        .appName('Block3MetaSpark')
        .master('local[*]')
        .getOrCreate()
    )

    raw_df = spark.read.text(str(input_path)).toDF('raw_json')

    UserProcessor = create_spark_processor(
        'UserProcessor',
        fields=[
            integer('user_id', required=True),
            string('email', required=True, coercion=normalize_email),
            integer('age', required=True, validator=validate_age),
            string('country', required=True, validator=validate_country),
        ],
        input_col='raw_json',
        output_col='data',
    )

    proc = UserProcessor()
    enforced = proc.enforce_schema(raw_df)
    validated = proc.with_validation_udf(enforced)

    print('=== AFTER SCHEMA ENFORCEMENT (SPARK) ===')
    enforced.select('raw_json', 'data', 'is_schema_ok', 'schema_error').show(truncate=False)

    print('=== AFTER VALIDATION UDF (SPARK) ===')
    validated.select(
        'raw_json', 'data', 'is_schema_ok', 'schema_error', 'is_valid', 'validation_error', 'validation.normalized'
    ).show(truncate=False)

    good = validated.filter(F.col('is_schema_ok') & F.col('is_valid')).select('data.*')
    bad = validated.filter(~(F.col('is_schema_ok') & F.col('is_valid'))).select(
        'raw_json', 'schema_error', 'validation_error'
    )

    print('=== GOOD RECORDS (SPARK) ===')
    good.show(truncate=False)

    print('=== BAD RECORDS (SPARK) ===')
    bad.show(truncate=False)

    print('=== QUALITY REPORT (SPARK) ===')
    total = validated.count()
    good_count = good.count()
    bad_count = bad.count()
    print(f'Total records: {total}')
    print(f'Valid records: {good_count}')
    print(f'Invalid records: {bad_count}')
    print(f'Quality score: {good_count / total * 100:.2f}%')

    print('Errors by schema_error:')
    validated.groupBy('schema_error').count().orderBy(F.desc('count')).show(truncate=False)

    print('Errors by validation_error:')
    validated.groupBy('validation_error').count().orderBy(F.desc('count')).show(truncate=False)

    spark.stop()

=== AFTER SCHEMA ENFORCEMENT (SPARK) ===
+-----------------------------------------------------------------------------+-----------------------------------+------------+-----------------------------------+
|raw_json                                                                     |data                               |is_schema_ok|schema_error                       |
+-----------------------------------------------------------------------------+-----------------------------------+------------+-----------------------------------+
|{"user_id": "123", "email": "  A@B.com  ", "age": "30", "country": "PL"}     |{NULL,   A@B.com  , NULL, PL}      |false       |MISSING_REQUIRED_FIELD_OR_TYPE_CAST|
|{"user_id": 124, "email": "X@Y.com", "age": 40, "country": "DE"}             |{124, X@Y.com, 40, DE}             |true        |NULL                               |
|{"user_id": "oops", "email": "bad@z.com", "age": "NaN", "country": "PL"}     |{NULL, bad@z.com, NULL, PL}        |false       |MISSIN